# fine tunning

In [1]:
import numpy as np
import pandas as pd
import sklearn
import pyarrow as pa
import torch

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PyArrow:", pa.__version__)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\farha\anaconda3\envs\farhan_torch_env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\farha\anaconda3\envs\farhan_torch_env\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\farha\anaconda3\envs\farhan_torch_env\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    s

AttributeError: _ARRAY_API not found

c:\Users\farha\anaconda3\envs\farhan_torch_env\Lib\site-packages\pandas\core\arrays\masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


NumPy: 2.2.6
Pandas: 3.1.1
Scikit-learn: 1.9.1
PyArrow: 26.0.1
PyTorch: 2.7.1+cu118
CUDA: True


In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datasets

from sentence_transformers import SentenceTransformer,InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

import json,warnings
warnings.filterwarnings('ignore')

print("Import Done")


c:\Users\farha\anaconda3\envs\farhan_torch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Import Done


C:\Users\farha\AppData\Local\Temp\ipykernel_10504\3001511593.py:7: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer,InputExample, losses
C:\Users\farha\AppData\Local\Temp\ipykernel_10504\3001511593.py:8: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator


In [3]:
df=pd.read_csv("cleaned_data.csv")
print(f"Loaded: {len(df)} pairs")
print(f"\nLabel distribution:")
print(df["match_label"].value_counts())
print(f"\nScore range: {df['match_score'].min():2f}-{df['match_score'].max():2f}")
df.head()

Loaded: 280 pairs

Label distribution:
match_label
medium    110
high       93
low        77
Name: count, dtype: int64

Score range: 0.060000-0.940000


,resume_text,job_description,match_label,match_score,match_label_lower,resume_len,jd_len
0,BI Analyst with 7 years of experience based in...,Fathom is hiring a Financial Analyst (Berlin)....,medium,0.53,medium,75,71
1,Data Scientist with 13 years of experience bas...,Lumen is hiring a Senior UX Designer (Singapor...,low,0.26,low,79,74
2,Software Engineer with 8 years of experience b...,Acme Corp is hiring a Product Manager (Seattle...,low,0.31,low,71,80
3,Backend Engineer with 15 years of experience b...,Umbrella Labs is hiring a Senior Data Scientis...,medium,0.46,medium,70,83
4,Cloud Engineer with 5 years of experience base...,Cobalt is hiring a Platform Engineer (Austin)....,high,0.77,high,69,70


In [4]:
train_df,temp_df=train_test_split(
    df,test_size=0.3,random_state=42,stratify=df["match_label"]
)
val_df,test_df=train_test_split(
    temp_df,test_size=0.5,random_state=42,stratify=temp_df["match_label"]
)
print(f"train: {len(train_df)}")
print(f"validation: {len(val_df)}")
print(f"test: {len(test_df)}")
print()

train: 196
validation: 42
test: 42



In [5]:
# verify each split has all three label
for name,split in[('Train',train_df),('Test',test_df),('Validation',val_df)]:
    print(f"{name} label dist: {split['match_label'].value_counts()}")

Train label dist: match_label
medium    77
high      65
low       54
Name: count, dtype: int64
Test label dist: match_label
medium    17
high      14
low       11
Name: count, dtype: int64
Validation label dist: match_label
medium    16
high      14
low       12
Name: count, dtype: int64


In [6]:
# convert to input format for sentence transformer
# text1=resume, text2=job_description label=match_score
train_example=[
    InputExample(texts=[row["resume_text"],row["job_description"]],label=float(row["match_score"])) for _,row in train_df.iterrows()
]

val_example=[
    InputExample(texts=[row["resume_text"],row["job_description"]],label=float(row["match_score"])) for _,row in val_df.iterrows()
]

In [7]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

PyTorch: 2.7.1+cu118
CUDA Available: True
CUDA Version: 11.8
GPU: NVIDIA GeForce RTX 2050
GPU Memory: 3.99951171875 GB


In [8]:
print(f"Loading base bert model...")
base_bert_model=SentenceTransformer('all-mpnet-base-v2',device='cuda')
print(f"Successfully loaded!")

Loading base bert model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3619.40it/s]


Successfully loaded!


In [17]:
# Evaluate base model on test set before fine-tuning
print("Evaluating base model on test set....")

base_preds = []

for _, row in test_df.iterrows():

    emb1 = base_bert_model.encode(row['resume_text'])
    emb2 = base_bert_model.encode(row['job_description'])

    sim = cosine_similarity([emb1], [emb2])[0][0]

    base_preds.append(float(sim))

base_mae = mean_absolute_error(test_df['match_score'], base_preds)

base_rmse = np.sqrt(
    mean_squared_error(test_df['match_score'], base_preds)
)
print(f'\nBase Model -Test set performance:')
print(f"Base Model MAE: {base_mae:.4f}")
print(f"Base Model RMSE: {base_rmse:.4f}")

Evaluating base model on test set....

Base Model -Test set performance:
Base Model MAE: 0.1946
Base Model RMSE: 0.2332


In [10]:
# Load base model
model = SentenceTransformer('all-mpnet-base-v2', device='cuda')

# Create training dataloader
train_dataloader = DataLoader(
    train_example,
    shuffle=True,
    batch_size=16
)

# Define training loss
train_loss = losses.CosineSimilarityLoss(model)

# Create validation evaluator
evaluator = EmbeddingSimilarityEvaluator.from_input_examples(
    val_example,
    name='ats-val'
)
print(f"Training setup ready")
print(f"Batch size: 16")
print(f"Train pairs; {len(train_example)}")
print(f"steps/epoch: {len(train_dataloader)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9039.06it/s]


Training setup ready
Batch size: 16
Train pairs; 196
steps/epoch: 13


In [11]:
total_steps=len(train_dataloader)*10
warmup_steps=int(total_steps*0.1)
print(f"total steps: {total_steps}")
print(f"warmup steps: {warmup_steps}")
print(f"\nStarting fine tunning....")

model.fit(
    train_objectives=[(train_dataloader,train_loss)],
    evaluator=evaluator,
    epochs=10,
    evaluation_steps=len(train_dataloader),
    warmup_steps=warmup_steps,
    output_path='models/finetuned-bert',
    save_best_model=True,
    use_amp=True,
    show_progress_bar=True
)
print(f"\nFine Tunning Done!")

total steps: 130
warmup steps: 13

Starting fine tunning....


Step,Training Loss,Validation Loss,Ats-val Pearson Cosine,Ats-val Spearman Cosine
13,No log,No log,0.721759,0.709102
26,No log,No log,0.749205,0.741131
39,No log,No log,0.764583,0.765457
52,No log,No log,0.771242,0.756213
65,No log,No log,0.775546,0.755970
78,No log,No log,0.767963,0.760268
91,No log,No log,0.781817,0.769349
104,No log,No log,0.786195,0.771214
117,No log,No log,0.786441,0.776566
130,No log,No log,0.786325,0.776566


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.35it/s]



Fine Tunning Done!


In [12]:
print(f"Loading fine tunned model")
finetuned_model = SentenceTransformer(
    "models/finetuned-bert",
    device="cuda"
)
print(f"Loaded fine tune model")

Loading fine tunned model


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3897.71it/s]


Loaded fine tune model


In [14]:
# Evaluate base model on test set before fine-tuning
print("Evaluating fine tunned base model on val set....")

ft_preds = []

for _, row in test_df.iterrows():

    emb1 = finetuned_model.encode(row['resume_text'])
    emb2 = finetuned_model.encode(row['job_description'])

    sim = cosine_similarity([emb1], [emb2])[0][0]

    ft_preds.append(float(sim))

ft_mae = mean_absolute_error(test_df['match_score'], ft_preds)

ft_rmse = np.sqrt(
    mean_squared_error(test_df['match_score'], ft_preds)
)
print(f'\nFined tunned Base Model -Test set performance:')
print(f"Base Model MAE: {ft_mae:.4f}")
print(f"Base Model RMSE: {ft_rmse:.4f}")

Evaluating fine tunned base model on val set....

Fined tunned Base Model -Test set performance:
Base Model MAE: 0.0921
Base Model RMSE: 0.1179


In [18]:
# comparison
print("="*40)
print("Model Comparison")
print("="*40)
print(f"Base MAE: {base_mae:.4f}")
print(f'FineTune MAE: {ft_mae:.4f}')


Model Comparison
Base MAE: 0.1946
FineTune MAE: 0.0921


In [22]:
metadata={
    'base_model':'all-mpnet-base-v2',
    'dataset':'merged_dataset_clean.csv',
    'total_pairs':len(df),
    'train_pairs':len(train_df),
    'test_pairs':len(test_df),
    'val_pairs':len(val_df),
    'epochs':10,
    'batch_size':16,
    'base_mae':round(float(base_mae),4),
    'finetuned_mae':round(float(ft_mae),4) 
}
with open('models/finetuned-bert/metadata','w')as f:
    json.dump(metadata,f,indent=2)
print(f"Saved: models/finetuned-bert/")
print(json.dumps(metadata,indent=2))


Saved: models/finetuned-bert/
{
  "base_model": "all-mpnet-base-v2",
  "dataset": "merged_dataset_clean.csv",
  "total_pairs": 280,
  "train_pairs": 196,
  "test_pairs": 42,
  "val_pairs": 42,
  "epochs": 10,
  "batch_size": 16,
  "base_mae": 0.1946,
  "finetuned_mae": 0.0921
}
